In [1]:
import pandas as pd
import os 
from pathlib import Path

In [2]:
folder = Path(r"C:\Users\ercj\Desktop\plain_language_summarization_platform\amia_questions_combined\annotations")
csv_files = list(folder.glob("*.csv"))
print(csv_files)
dfs = [pd.read_csv(p) for p in csv_files]

[WindowsPath('C:/Users/ercj/Desktop/plain_language_summarization_platform/amia_questions_combined/annotations/Question Annotation Study (Responses) - Form Responses 1.csv'), WindowsPath('C:/Users/ercj/Desktop/plain_language_summarization_platform/amia_questions_combined/annotations/Question Annotation Study 2 (Responses) - Form Responses 1.csv'), WindowsPath('C:/Users/ercj/Desktop/plain_language_summarization_platform/amia_questions_combined/annotations/Question Annotation Study 3 (Responses) - Form Responses 1.csv'), WindowsPath('C:/Users/ercj/Desktop/plain_language_summarization_platform/amia_questions_combined/annotations/Question Annotation Study 4 (Responses) - Form Responses 1.csv'), WindowsPath('C:/Users/ercj/Desktop/plain_language_summarization_platform/amia_questions_combined/annotations/Question Annotation Study 5 (Responses) - Form Responses 1.csv'), WindowsPath('C:/Users/ercj/Desktop/plain_language_summarization_platform/amia_questions_combined/annotations/Question Annotati

In [ ]:
all_abstract_dfs = []

for csv_path in csv_files:
    df = pd.read_csv(csv_path)
    overall = df[df["Email address"] == "Overall"]

    if overall.empty:
        continue

    cog_cols = [
        c for c in overall.columns
        if (
            "For Question" in c
            and "cognitive" in c.lower()
        )
    ]

    long = overall[cog_cols].melt(
        var_name="raw_col",
        value_name="cognitive_level"
    )

    long["abstract_id"] = long["raw_col"].str.split("_", n=1).str[0]
    long["question"] = (
        long["raw_col"]
        .str.extract(r"Question (\d+)")
        .astype(int)
    )

    per_file_df = long.pivot_table(
        index="abstract_id",
        columns="question",
        values="cognitive_level",
        aggfunc="first"
    )

    per_file_df.columns = [f"Q{c}" for c in per_file_df.columns]

    comment_cols = [
        c for c in overall.columns
        if (
            "If any questions or answer choices need revision" in c
        )
    ]

    if comment_cols:
        comments_long = overall[comment_cols].melt(
            var_name="raw_col",
            value_name="revision_comment"
        )

        comments_long["abstract_id"] = (
            comments_long["raw_col"].str.split("_", n=1).str[0]
        )

        comments_df = comments_long.set_index("abstract_id")[["revision_comment"]]
        per_file_df = per_file_df.join(comments_df, how="left")

    all_abstract_dfs.append(per_file_df)

final_df = pd.concat(all_abstract_dfs).sort_index()

In [20]:
master_df = pd.read_csv(r"C:\Users\ercj\Desktop\plain_language_summarization_platform\edited_personalizing_dataset.csv")
final_df = final_df.reset_index()

master_df["abstract_id"] = master_df["abstract_id"].astype(str)
final_df["abstract_id"] = final_df["abstract_id"].astype(str)

merged_df = master_df.merge(final_df, on="abstract_id", how="left")

In [21]:
merged_df.to_csv("merged_final_abstracts.csv", index=False)